# Statevector snapshot comparison

Statevector snapshots reduce the cost of simulating $n$ time steps of a QBM from quadratic to linear.

Without snapshots, time step $k$ is simulated from scratch: prepare the initial conditions, then apply the
time step circuit $k$ times. Covering steps $0 \dots n$ therefore costs $\frac{n(n+1)}{2}$ applications of
the algorithm, and the last circuit is $n$ time steps deep.

With snapshots, `qlbm` keeps the statevector produced at step $k$ and hands it straight back to the
simulation as the *initial state* of step $k + 1$ (`Block.statevector(initial_state=psi)`).
Every step then costs exactly one application of the algorithm, and one command stream serves all $n$
iterations. What happens between two time steps is the job of `qlbm`'s *reinitialization* interface, which
is where algorithm-specific state carrying — such as the space-time reinitializer — plugs in.

Because snapshots need a full copy of the underlying statevector, the technique is feasible only on
simulators. It is toggled through the `statevector_snapshots` parameter of `Runner.run`.

In this notebook, we measure what that toggle is worth.

In [ ]:
import logging.config
import re
from logging import Logger, getLogger

from qlbm.components import (
    CQLBM,
    EmptyPrimitive,
    GridMeasurement,
    MSInitialConditions,
)
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import MSLattice
from qlbm.tools.utils import create_directory_and_parents, get_circuit_properties

In [ ]:
# Import analysis and plotting utilities
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme()

## Step 1: Setup

Before benchmarking the simulations, we first have to choose some behaviour to simulate and parameters for
our quantum circuits. The only difference between the two sessions below is the `statevector_snapshots`
argument handed to `QarpRunner.run`; everything else — lattices, circuits, shot budget — is identical.

In [ ]:
def benchmark(
    lattice_dicts,
    num_shots: int,
    num_steps: int,
    statevector_snapshots: bool,
    logger: Logger,
    dummy_logger: Logger,
    root_output_dir: str,
    num_repetitions: int = 5,
) -> None:
    """Simulate every lattice for `num_steps` steps, with or without snapshots."""
    for rep in range(num_repetitions):
        logger.info(f"Repetition #{rep + 1} of {num_repetitions}")
        for count, lattice_dict in enumerate(lattice_dicts):
            logger.info(f"Combination #{count + 1} of {len(lattice_dicts)}")

            lattice = MSLattice(lattice_dict, logger=dummy_logger)
            logger.info(
                f"Lattice={lattice.logger_name()}, num_qubits={lattice.num_total_qubits}"
            )

            output_dir = (
                f"{root_output_dir}/{lattice.logger_name()}-{statevector_snapshots}"
            )

            create_directory_and_parents(output_dir)

            cfg = SimulationConfig(
                initial_conditions=MSInitialConditions(lattice),
                algorithm=CQLBM(lattice),
                postprocessing=EmptyPrimitive(lattice),
                measurement=GridMeasurement(lattice),
                optimization_level=0,
                shots=num_shots,
                logger=dummy_logger,
            )

            cfg.validate()
            cfg.prepare_for_simulation()

            logger.info(
                f"Final circuit properties: {get_circuit_properties(cfg.algorithm)}"
            )

            # Create a runner object to simulate the circuit
            runner = QarpRunner(cfg, lattice, logger=logger, seed=1234)

            runner.run(
                num_steps,  # Number of time steps
                num_shots,  # Number of shots per time step
                output_dir,
                statevector_snapshots=statevector_snapshots,
            )

In [ ]:
NUM_SHOTS = 2**10
NUM_STEPS = 4
NUM_REPS = 2
ROOT_OUTPUT_DIR = "qlbm-output/benchmark-statevector-snapshots"
LOG_FILE = f"{ROOT_OUTPUT_DIR}/qlbm.log"

create_directory_and_parents(ROOT_OUTPUT_DIR)

# Truncate the log so that re-running the notebook does not append to a previous session
with open(LOG_FILE, "w"):
    pass

In [ ]:
lattices = [
    {
        "lattice": {"dim": {"x": 8, "y": 8}, "velocities": {"x": 4, "y": 4}},
        "geometry": [],
    },
    {
        "lattice": {"dim": {"x": 8, "y": 8}, "velocities": {"x": 4, "y": 4}},
        "geometry": [
            {"shape": "cuboid", "x": [5, 6], "y": [1, 2], "boundary": "specular"}
        ],
    },
    {
        "lattice": {"dim": {"x": 8, "y": 8}, "velocities": {"x": 4, "y": 4}},
        "geometry": [
            {"shape": "cuboid", "x": [5, 6], "y": [1, 2], "boundary": "specular"},
            {"shape": "cuboid", "x": [5, 6], "y": [5, 6], "boundary": "specular"},
        ],
    },
]

dummy_logger = getLogger("dummy")

# By logging at this point we ignore the output of circuit creation
logging.config.fileConfig("statevector_snapshots_logging.conf")
logger = getLogger("qlbm")

## Step 2: Simulation

Now that we have set up our simulations and configured logging, we can run them by calling the `benchmark`
function once per session. The parameters above are demo-sized (3 lattices of 15 qubits, 4 time steps,
2 repetitions) and the two cells together take a couple of minutes.

> **_CAUTION:_** Each cell below should be run exactly once per session, since both append to the log that
> Step 3 parses.

In [ ]:
logger.info("Session: snapshots=True")
benchmark(
    lattices,
    NUM_SHOTS,
    NUM_STEPS,
    True,
    logger,
    dummy_logger,
    ROOT_OUTPUT_DIR,
    num_repetitions=NUM_REPS,
)

In [ ]:
logger.info("Session: snapshots=False")
benchmark(
    lattices,
    NUM_SHOTS,
    NUM_STEPS,
    False,
    logger,
    dummy_logger,
    ROOT_OUTPUT_DIR,
    num_repetitions=NUM_REPS,
)

## Step 3: Analysis

Once the simulations have concluded, the performance logs created by `qlbm` will be under
`qlbm-output/benchmark-statevector-snapshots/qlbm.log`. The runner logs how long simulating each time step
took, and the benchmark logs the algorithm's gate count once per lattice. Without snapshots, step $k$
applies the algorithm $k$ times; with snapshots, every step after the first applies it once. A single pass
over the file turns those lines into a dataframe.

In [ ]:
SESSION_RE = re.compile(r"Session: snapshots=(True|False)")
REPETITION_RE = re.compile(r"Repetition #(\d+) of")
LATTICE_RE = re.compile(r"Lattice=(\S+), num_qubits=(\d+)")
GATES_RE = re.compile(r"Final circuit properties: \('QARP', \d+, \d+, (\d+)\)")
DURATION_RE = re.compile(r"Simulation of (\d+) steps took (\d+) \(ns\)")

with open(LOG_FILE, "r") as f:
    lines = f.readlines()

records = []
snapshots, repetition, lattice_name, num_qubits, gates = None, None, None, None, None

for line in lines:
    match = SESSION_RE.search(line)
    if match:
        snapshots = match.group(1) == "True"
        continue

    match = REPETITION_RE.search(line)
    if match:
        repetition = int(match.group(1))
        continue

    match = LATTICE_RE.search(line)
    if match:
        lattice_name, num_qubits = match.group(1), int(match.group(2))
        continue

    match = GATES_RE.search(line)
    if match:
        gates = int(match.group(1))
        continue

    match = DURATION_RE.search(line)
    if match:
        step = int(match.group(1))
        records.append(
            {
                "Lattice": lattice_name,
                "Dimensions": lattice_name.split("-")[1],
                "Obstacles": int(lattice_name.split("-")[2]),
                "Circuit Qubits": num_qubits,
                "Repetition": repetition,
                "Step": step,
                "Commands": gates * (min(step, 1) if snapshots else step),
                "Duration (ns)": int(match.group(2)),
                "Snapshots": snapshots,
            }
        )

df = pd.DataFrame.from_records(records)
df["Duration (s)"] = df["Duration (ns)"] / 1e9
df["Cumulative Duration (s)"] = df.groupby(
    ["Snapshots", "Lattice", "Repetition"], sort=False
)["Duration (s)"].cumsum()
df

In [ ]:
ax = sns.lineplot(
    df,
    x="Step",
    y="Cumulative Duration (s)",
    hue="Obstacles",
    style="Snapshots",
    markers=True,
)
plt.xticks(pd.unique(df["Step"]))
plt.title("Cumulative Simulation Time")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

The cumulative curves diverge exactly as the operation counts predict: without snapshots, step $k$ replays
the algorithm $k$ times, so the circuit handed to the simulator grows linearly with the step index. With
snapshots it stays the same size forever.

In [ ]:
ax = sns.lineplot(
    df,
    x="Step",
    y="Commands",
    hue="Obstacles",
    style="Snapshots",
    markers=True,
)
plt.xticks(pd.unique(df["Step"]))
plt.title("Simulated Commands per Time Step")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))